# Multipath reflections on the buoys: what they are, when they appear, what they tell us

Phase 1 pinger arrivals on the two buoys (Rx902, Rx01), checked against the EMODnet bathymetry; phase 2
boat noise at the end. Inputs: the cached compressed pulses `out/multipath/arrivals_<rx>.npz`, the ridge
diagnostics in `out/multipath/mp_range_phase1.npz`, and the phase 1 track. Ray tracing: `processing/bathy.py`.

Everything is read from `processing/forfun.toml` and the pipeline outputs; `processing/pipeline.py`
re-executes this notebook when either changes. The findings below quote the numbers of the run of
2026-10-21; where they differ, the cell outputs are current.

**Findings**

1. **The water is much shallower than the earlier multipath notes assumed.** EMODnet gives 13.3 m at Rx902,
   12.9 m at Rx01, and 5.8–9.3 m under the pinger track, with a shoal of 6–7 m between ~330 and 570 s.
   - So the pinger cannot be at its nominal 10 m, and the hydrophones (nominal 10 m) sit ~3 m above the seabed.
   - The "water depth 24–27 m" in `notes/D_multipath.md` came from reading the late arrival as a single
     bottom bounce, and is wrong.
   - The coherent fit there searched D = 12–45 m, so the true depths were off its grid. It "ran to the
     shallow edge", which is what a shallower true depth would do.
2. **What the arrivals look like.**
   - Behind the direct path there is an early cluster at 0.05–0.6 ms (S, B, SB) carrying about as much
     energy as the direct path: ≈ 0 dB on Rx902, ≈ +3 dB on Rx01.
   - There is also one discrete late arrival per buoy at 0.5–1.7 ms. It comes and goes, and at times it is
     as strong as the direct path.
   - Plotted against τ·R it is flat, so it is a real geometric path with delay ∝ 1/R: τR = 0.30–0.43 m·s on
     Rx902, and 0.39 or 0.72–0.81 m·s on Rx01. With the charted depths it is not "the bottom echo":
     of the paths with at most two reflections (the only ones computed) only BS comes this late.
   - Single chirps show it directly (section 3): the envelope fades every 10–25 ms along the sweep
     where the late copy is strong, and stays smooth over the shoal.
   - A frequency shift between the echo and the direct chirp cannot be measured: on an LFM chirp it trades
     with delay (10 µs per Hz), and the geometry predicts less than 0.1 Hz (section 3).
3. **When it appears.**
   - It is there while the water under the pinger is deeper than ~7.5 m (52–325 s and 575–775 s).
   - It vanishes over the shoal. For Rx902, 0 % of pings in water under 7.25 m have it, against 50–75 % in
     deeper water.
   - The shoal is also where the run 1 leg turns around, so position and depth are confounded.
4. **No path with at most two reflections explains it.** Ray tracing over the EMODnet bottom, BS is the only
   candidate, and its best fit sits at the grid edge ($z_s$ = 1 m, $z_r$ = 12.5 m, 0.8 m above the seabed at
   Rx902) with a log-delay misfit of 0.33 (Rx902) and 0.48 (Rx01), i.e. delays off by 40–60 %. Either the late
   arrival has more than two reflections, or it is not a specular surface/bottom path (e.g. a scatterer or
   a reflection off the shoal's side). The sloping bottom does not explain the within-segment drift better
   than plain 1/R.
5. **Phase 2:** the boat noise shows no multipath lags. With the source near the surface and the hydrophone
   near the bottom, the predicted delays are under ~1.5 ms, inside the stationary part of the
   autocorrelation.

> **Correction (2026-10-22).** The hydrophones are moored **10 m above the seabed**, not at 10 m depth
> (operator). With the charted depth and the 2026-10-20 tide that is 3.28 m (Rx902) and 2.65 m (Rx01) below the
> surface (`config.py`, `height_above_seabed`). The figures below are re-executed with these depths. The findings
> text in this notebook was written for 10 m and is out of date where it quotes hydrophone depths, path families or
> fits: with hydrophones near the surface the bottom path B becomes separable and the late arrival near 0.77 ms lies
> beyond every path with at most two reflections. See `multipath_paths.ipynb` (findings) and `processing/notes/I_multipath_paths.md`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "processing"))

In [ ]:
import bathy
import config
import multipath_range as mp
import stamp

cfg = config.load()
T = np.load(ROOT / "out" / "toa_phase1.npz")
St = np.load(ROOT / "out" / "phase1" / "state.npz")
M = np.load(ROOT / "out" / "multipath" / "mp_range_phase1.npz")
pe = mp.load_track(T)  # pinger ENU per ping (graph solve)
t_ping = St["t0"] + T["k"] * St["Tp"]  # emission time, buoy clock [s]
buoys = dict(zip(mp.BUOYS, St["buoys"]))  # graph-estimated buoy positions
usbl = next(r.en for r in cfg.receivers if r.kind == "usbl")
col = {n: T["receivers"].tolist().index(n) for n in mp.BUOYS}
arr = {
    n: mp.extract(n) for n in mp.BUOYS
}  # cached compressed pulses around the direct path
if not (bathy.BATHY_DIR / "E6_2024.tif").exists():
    raise FileNotFoundError(
        "Bathymetry file not found. Please download it from the repository and place it in the bathy directory."
    )
B = bathy.Bathymetry(cfg)
C = float(dict(zip(St["names"], St["globals"]))["c"])  # phase 1 posterior sound speed
Z_RX = {n: cfg.receiver(n).depth for n in mp.BUOYS}  # configured hydrophone depths
R = {n: np.hypot(*(pe - buoys[n]).T) for n in mp.BUOYS}
D_pinger = B.depth(pe[:, 0], pe[:, 1])
runs = [(r.name, r.start, r.end) for r in cfg.runs if r.pinger]
tr = np.genfromtxt(ROOT / "out" / "phase1" / "track.csv", delimiter=",", names=True)
course = np.full(len(t_ping), np.nan)
course[np.searchsorted(T["k"], tr["ping"].astype(int))] = (
    np.degrees(np.arctan2(tr["v_n"], tr["v_e"])) % 360
)
for n in mp.BUOYS:
    print(
        f"{n}: water depth {B.depth(*buoys[n]):.1f} m at the buoy, range to pinger "
        f"{np.nanmin(R[n]):.0f}-{np.nanmax(R[n]):.0f} m, {arr[n]['ok'].sum()} pings with a compressed pulse"
    )
print(
    f"water depth under the pinger: {np.nanmin(D_pinger):.1f}-{np.nanmax(D_pinger):.1f} m"
)

## 1. The site: shallow water, a shoal under the track

EMODnet mean-depth DTM (1/16 arcmin, ~115 × 84 m cells; around here it is the Croatian 2004 survey,
`croatia2004`), bilinear-interpolated. Grey is land: the Vodice shore to the north and an islet south of Rx01.

In [ ]:
e = np.arange(-650, 250, 5.0)
nn = np.arange(-900, 150, 5.0)
E, N = np.meshgrid(e, nn)
Zd = B.depth(E, N)
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw=dict(width_ratios=[1, 1.3]))
im = ax.pcolormesh(E, N, np.where(Zd > 0, Zd, np.nan), cmap="Blues", vmin=0, vmax=20, shading="auto")
ax.contourf(E, N, Zd, levels=[-100, 0], colors=["0.75"])
cs = ax.contour(E, N, Zd, levels=[5, 8, 10, 12, 14, 16], colors="k", linewidths=0.5)
ax.clabel(cs, fmt="%g m", fontsize=7)
sc = ax.scatter(pe[:, 0], pe[:, 1], c=t_ping, s=3, cmap="viridis")
for n, m in zip(mp.BUOYS, ("^", "v")):
    ax.plot(*buoys[n], m, color="C3", ms=9)
    ax.annotate(n, buoys[n], xytext=(6, 6), textcoords="offset points", color="C3")
ax.plot(*usbl, "s", color="C1", ms=8)
ax.annotate("USBL", usbl, xytext=(6, 6), textcoords="offset points", color="C1")
ax.set_aspect("equal")
ax.set_xlabel("east [m]")
ax.set_ylabel("north [m]")
fig.colorbar(im, ax=ax, label="water depth [m]", shrink=0.7)
fig.colorbar(sc, ax=ax, label="pinger track: buoy time [s]", shrink=0.7, location="bottom", pad=0.12)
ax.set_title("EMODnet depth, pinger track, receivers")
ax2.plot(t_ping, D_pinger, "k", lw=1, label="under the pinger")
for n in mp.BUOYS:
    ax2.axhline(B.depth(*buoys[n]), color="C3", ls="--" if n == "Rx01" else "-", lw=1, label=f"at {n}")
ax2.axhline(cfg.pinger.depth, color="C1", lw=2, alpha=0.6, label=f"configured pinger depth ({cfg.pinger.depth:g} m)")
ax2.axhline(Z_RX["Rx902"], color="C2", lw=2, alpha=0.6, label=f"configured hydrophone depth ({Z_RX['Rx902']:g} m)")
ax2.invert_yaxis()
ax2.set_xlabel("buoy time [s]")
ax2.set_ylabel("depth [m]")
ax2.legend(loc="lower left", fontsize=8)
ax2.set_title("water depth along the run")
fig.tight_layout()

## 2. Which reflections to expect

In shallow water every source–receiver pair has a ladder of image paths: S (one surface bounce),
B (one bottom bounce), SB, BS, SBS, BSB and so on. The ray tracer `bathy.eigenrays` follows straight rays
over the sloping EMODnet bottom between the pinger and the buoy (on a flat bottom it reproduces the
image method in `multipath_range.py`; see `tests/test_bathy.py`). Only paths with at most two reflections
are computed (`bathy.MAX_BOUNCES`): S, B, SB and BS.

Both the pinger depth $z_s$ and the hydrophone depth $z_r$ are unknown. A 10 m tow depth does not fit
under the track (6–9 m of water). This example uses the configured depths, `[pinger].depth` and
`[[receiver]].depth` (printed below). The compressed pulse
resolves about $1/B$ = 0.1 ms.

In [ ]:
FAMS = ["S", "B", "SB", "BS"]  # every path with at most bathy.MAX_BOUNCES = 2 reflections
ZS_EX, ZR_EX = cfg.pinger.depth, Z_RX["Rx902"]  # example geometry: the configured depths
print(f"example depths: z_s = {ZS_EX:g} m ([pinger].depth), z_r = {ZR_EX:g} m ([[receiver]].depth)")
seg0 = M["mp_ridge_seg"][:, col["Rx902"]]
k_ex = int(np.median(np.flatnonzero(seg0 == 0)))          # a ping inside the first Rx902 ridge segment
fig, axs = plt.subplots(2, 1, figsize=(14, 7))
for ax, n in zip(axs, mp.BUOYS):
    x, zb = B.profile(pe[k_ex], buoys[n])
    rays = bathy.eigenrays(ZS_EX, ZR_EX, x, zb)
    L0 = rays[0]["L"]
    ax.fill_between(x, zb, zb.max() + 2, color="0.8")
    print(f"{n}, ping {T['k'][k_ex]} (t = {t_ping[k_ex]:.0f} s), R = {x[-1]:.0f} m")
    for i, r in enumerate(rays):
        p = bathy.ray_path(ZS_EX, r["angle"], x, zb)
        lab = bathy.family(r)
        ax.plot(p[:, 0], p[:, 1], lw=1, label=f"{lab}: {(r['L'] - L0) / C * 1e3:.2f} ms")
        print(f"   {lab:6s} {r['ns']} surface, {r['nb']} bottom, delay {(r['L'] - L0) / C * 1e3:6.3f} ms, "
              f"launch {r['angle']:+5.1f} deg")
    ax.invert_yaxis()
    ax.set_ylim(zb.max() + 1, -0.5)
    ax.set_ylabel("depth [m]")
    ax.set_title(f"{n}: eigenrays at t = {t_ping[k_ex]:.0f} s (z_s = {ZS_EX} m, z_r = {ZR_EX} m; "
                 "vertical exaggeration ~20x)")
    ax.legend(ncol=5, fontsize=8, loc="lower center")
axs[-1].set_xlabel("horizontal distance from the pinger [m]")
fig.tight_layout()

## 3. One ping, close up

The compressed pulse envelope behind the direct path (cached by `multipath_range.extract`: η-matched replica,
aligned on the direct-path TOA). Thin: this ping. Thick: incoherent mean over ±5 pings. Ticks: the
predicted delays from section 2.

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, n in zip(axs, mp.BUOYS):
    d = arr[n]
    A, _ = mp.echogram(d, smooth=1)
    ks = [k for k in range(k_ex - 5, k_ex + 6) if d["ok"][k]]
    ax.plot(d["t"] * 1e3, 10 * np.log10(A[k_ex] + 1e-9), lw=0.6, color="0.5", label=f"ping {T['k'][k_ex]}")
    ax.plot(d["t"] * 1e3, 10 * np.log10(A[ks].mean(0) + 1e-9), lw=1.5, color="k", label=f"mean of {len(ks)} pings")
    x, zb = B.profile(pe[k_ex], buoys[n])
    tau = bathy.family_delays(ZS_EX, ZR_EX, x, zb, FAMS, C)
    for i, (f, t) in enumerate(zip(FAMS, tau)):
        if np.isfinite(t):
            ax.axvline(t * 1e3, color=f"C{i}", lw=1, alpha=0.8)
            ax.text(t * 1e3, 1, f, color=f"C{i}", fontsize=8, ha="center", va="bottom")
    tr_ = M["mp_tau_ridge"][k_ex, col[n]]
    if np.isfinite(tr_):
        ax.plot(tr_ * 1e3, -3, "v", color="C3", ms=10, label="tracked late arrival (ridge)")
    ax.set_ylim(-45, 5)
    ax.set_ylabel("power rel. direct [dB]")
    ax.set_title(f"{n}, R = {R[n][k_ex]:.0f} m")
    ax.legend(loc="upper right", fontsize=8)
axs[-1].set_xlim(-0.3, 4)
axs[-1].set_xlabel("delay after the direct path [ms]")
fig.tight_layout()

### Individual chirps as recorded

Three pings, picked automatically: for each buoy the ping where the tracked late arrival is strongest,
and a strong Rx902 ping over the shoal, where there is none.

- **Left: the band-passed recording.** The chirp sweeps linearly from 30 to 40 kHz, so time along the pulse is
  also frequency (top axis). A copy delayed by τ interferes with the direct chirp and cancels it wherever the two
  are out of phase. The envelope therefore fades at intervals of 1/τ in frequency, i.e. every 10 ms / (τ in ms)
  along the pulse (about 13 ms for τ = 0.77 ms). Without a strong late path the envelope stays smooth.
- **Right: the same ping after the matched filter.** Each path becomes a peak at its delay. Dashed lines: the
  ray-traced delays of the paths with at most two reflections, for the example depths of section 2.


In [ ]:
from scipy import signal as sps

import recordings


def pick_chirps():
    """(buoy, row, why): the strongest late arrival per buoy, and a strong ping over the shoal without one."""
    out = []
    for n in mp.BUOYS:
        con = np.where(M["mp_ridge_seg"][:, col[n]] >= 0, M["mp_ridge_contrast"][:, col[n]], -np.inf)
        out.append((n, int(np.argmax(con)), "strongest late arrival"))
    shoal = arr["Rx902"]["ok"] & (D_pinger < 7.0) & (M["mp_ridge_seg"][:, col["Rx902"]] < 0)
    k = np.flatnonzero(shoal)[np.argmax(T["snr"][shoal, col["Rx902"]])]
    out.append(("Rx902", int(k), "over the shoal, no late arrival"))
    return out


def raw_chirp(n, k, pre=0.004, post=0.006):
    """Band-passed recording around ping row k on buoy n: t [s rel. TOA], analytic signal."""
    det = np.load(ROOT / "out" / f"detections_{n}.npz")["det"]
    i = T["det_index"][k, col[n]]
    t_rx, eta = det["t_rx"][i], det["eta"][i]
    rec = recordings.load(cfg.receiver(n))
    x, tx0 = rec.segment(t_rx - pre, t_rx + cfg.pinger.duration / eta + post, ch=0, clock="rx")
    x = np.asarray(x).ravel()
    sos = sps.butter(6, [cfg.pinger.f0 - 1500, cfg.pinger.f1 + 1500], "bandpass", fs=rec.fs, output="sos")
    xa = sps.hilbert(sps.sosfiltfilt(sos, x))
    return tx0 + np.arange(len(x)) / rec.fs - t_rx, xa


picks = pick_chirps()
fig, axs = plt.subplots(len(picks), 2, figsize=(16, 4 * len(picks)), gridspec_kw=dict(width_ratios=[1.8, 1]))
for row, (n, k, why) in zip(axs, picks):
    t, xa = raw_chirp(n, k)
    x_, zb = B.profile(pe[k], buoys[n])
    tau = bathy.family_delays(ZS_EX, ZR_EX, x_, zb, FAMS, C)
    title = f"{n}, ping {T['k'][k]} (t = {t_ping[k]:.0f} s, R = {R[n][k]:.0f} m, {D_pinger[k]:.1f} m under the pinger): {why}"
    # 1) the recording: multipath copies beat with the direct chirp, so the envelope fades along the sweep
    ax = row[0]
    env = np.abs(xa) / np.abs(xa).max()
    ax.plot(t * 1e3, xa.real / np.abs(xa).max(), lw=0.3, color="0.6")
    ax.plot(t * 1e3, env, lw=1, color="k", label="envelope")
    ax2 = ax.twiny()
    ax2.set_xlim(*(cfg.pinger.f0 + (np.array(ax.get_xlim()) * 1e-3) * (cfg.pinger.f1 - cfg.pinger.f0) / cfg.pinger.duration) / 1e3)
    ax2.set_xlabel("instantaneous chirp frequency [kHz]", fontsize=8)
    ax.set_xlabel("time after the direct-path TOA [ms]")
    ax.set_ylabel("band-passed, rel. max")
    ax.set_title(title, fontsize=9, loc="left")
    # 2) the compressed pulse of the same ping
    ax = row[1]
    A, _ = mp.echogram(arr[n], smooth=1)
    ax.plot(arr[n]["t"] * 1e3, 10 * np.log10(A[k] + 1e-9), color="k", lw=1)
    for i, (fam, tt) in enumerate(zip(FAMS, tau)):
        if np.isfinite(tt):
            ax.axvline(tt * 1e3, color=f"C{i}", lw=0.8, ls="--")
            ax.text(tt * 1e3, 2, fam, color=f"C{i}", fontsize=8, ha="center")
    tr_ = M["mp_tau_ridge"][k, col[n]]
    if np.isfinite(tr_):
        ax.plot(tr_ * 1e3, -3, "v", color="C3", ms=9, label="tracked late arrival")
        ax.legend(fontsize=8, loc="upper right")
    ax.set_xlim(-0.3, 2.2)
    ax.set_ylim(-35, 5)
    ax.set_xlabel("delay after direct [ms]")
    ax.set_ylabel("power rel. direct [dB]")
    ax.set_title("compressed pulse (matched filter)", fontsize=9)
fig.tight_layout()


### Frequency shift between the direct chirp and its echo

Each path leaves the pinger at its own vertical angle, so it can carry a slightly different Doppler shift.
Test on the two pings with the strongest late arrival:

1. Separate the paths. Gate the matched-filter output around the direct arrival (±0.08 ms; it still holds the
   S and B paths, which are 0.1 ms behind and unresolved) and around the tracked late echo (±0.15 ms), then
   re-synthesise each part (gated output convolved with the replica).
2. Spectrogram of the recorded chirp (all paths, where the dark gaps on the sweep are the fades of section 3)
   and of the separated echo (same colour scale).
3. Correlate the echo against the direct path for frequency shifts from −400 to +400 Hz.


In [ ]:
p_ = cfg.pinger
KR = (p_.f1 - p_.f0) / p_.duration                  # sweep rate [Hz/s]
FC = 0.5 * (p_.f0 + p_.f1)


def replica(t, eta):
    u = eta * t / p_.duration
    s = np.exp(2j * np.pi * (p_.f0 * p_.duration * u + 0.5 * KR * (p_.duration * u) ** 2))
    return np.where((u >= 0) & (u < 1), s, 0)


def split_paths(n, k, gates):
    """Recording of ping k split into paths by gating its matched-filter output in delay and
    re-synthesising (gated output convolved with the replica). gates: {name: (tau0, tau1) [s]}."""
    t, xa = raw_chirp(n, k)
    det = np.load(ROOT / "out" / f"detections_{n}.npz")["det"]
    eta = det["eta"][T["det_index"][k, col[n]]]
    fs = 1 / (t[1] - t[0])
    s = replica(np.arange(int(p_.duration / eta * fs) + 1) / fs, eta)
    N = int(2 ** np.ceil(np.log2(len(xa) + len(s))))
    S_ = np.fft.fft(s, N)
    c = np.fft.ifft(np.fft.fft(xa, N) * np.conj(S_))          # c[i] = MF output at lag t[0] + i/fs
    lag = t[0] + np.arange(N) / fs
    out = {}
    for name, (a, b) in gates.items():
        cg = np.where((lag >= a) & (lag < b), c, 0)
        out[name] = np.fft.ifft(np.fft.fft(cg, N) * S_)[:len(xa)] / np.sum(np.abs(s) ** 2)
    return t, xa, out, fs


def shift_correlation(xd, xl, t, fs, nus, max_lag=2.5e-3):
    """|correlation| of the echo xl, frequency-shifted by -nu, against the direct xd:
    rows nu, columns lag [s] (echo delay)."""
    L = int(max_lag * fs)
    N = int(2 ** np.ceil(np.log2(2 * len(xd))))
    D = np.conj(np.fft.fft(xd, N))
    out = []
    for nu in nus:
        r = np.fft.ifft(np.fft.fft(xl * np.exp(-2j * np.pi * nu * t), N) * D)
        out.append(np.abs(np.r_[r[-L:], r[:L + 1]]))
    return np.arange(-L, L + 1) / fs, np.array(out)


def range_rate(n):
    rr = np.gradient(R[n], t_ping)
    return np.convolve(np.nan_to_num(rr), np.ones(9) / 9, "same")


NUS = np.arange(-400.0, 400.1, 5.0)          # frequency shifts tried [Hz]
NPER = 512                                   # spectrogram window (5.3 ms at 96 kHz)
fig, axs = plt.subplots(2, 3, figsize=(17, 8.5), gridspec_kw=dict(width_ratios=[1, 1, 1.1]))
shift_rows = []
for row, (n, k, why) in zip(axs, [p for p in pick_chirps() if p[2] == "strongest late arrival"]):
    tr = M["mp_tau_ridge"][k, col[n]]
    t, xa, P, fs = split_paths(n, k, {"direct": (-0.08e-3, 0.08e-3), "echo": (tr - 0.15e-3, tr + 0.15e-3)})
    ref = None
    for ax, (lab, x) in zip(row[:2], (("recorded chirp (all paths)", xa), ("separated late echo", P["echo"]))):
        f, ts, Sx = sps.spectrogram(x, fs=fs, nperseg=NPER, noverlap=NPER * 7 // 8, return_onesided=False)
        f, Sx = np.fft.fftshift(f), np.fft.fftshift(Sx, axes=0)
        band = (f > p_.f0 - 2e3) & (f < p_.f1 + 2e3)
        ref = Sx[band].max() if ref is None else ref
        ax.pcolormesh((ts + t[0]) * 1e3, f[band] / 1e3, 10 * np.log10(Sx[band] / ref + 1e-12), vmin=-40, vmax=0,
                      cmap="magma", shading="nearest")
        ax.set_xlabel("time after the direct-path TOA [ms]")
        ax.set_ylabel("frequency [kHz]")
        ax.set_title(f"{n} ping {T['k'][k]}: {lab}", fontsize=9)
    # frequency-shift correlation of the echo against the direct path
    lags, X = shift_correlation(P["direct"], P["echo"], t, fs, NUS)
    Xd = 20 * np.log10(X / X.max())
    ax = row[2]
    ax.pcolormesh(lags * 1e3, NUS, Xd, vmin=-20, vmax=0, cmap="viridis", shading="nearest")
    i, j = np.unravel_index(X.argmax(), X.shape)
    ax.plot(lags[j] * 1e3, NUS[i], "r+", ms=12, mew=2)
    ax.plot((lags[j] + (NUS - NUS[i]) / KR) * 1e3, NUS, "w--", lw=0.8, label="delay + shift / sweep rate")
    ax.set_xlim(tr * 1e3 - 1.5, tr * 1e3 + 1.5)
    ax.set_xlabel("echo delay behind the direct path [ms]")
    ax.set_ylabel("frequency shift of the echo [Hz]")
    ax.set_title("correlation, echo vs direct (dB)", fontsize=9)
    ax.legend(fontsize=8, loc="lower right")
    prof = Xd.max(1)
    rr = range_rate(n)[k]
    # predicted extra shift of each path over the direct one: the pinger's horizontal velocity
    # projects onto each launch angle (receiver at rest; surface motion not included)
    rays = {bathy.family(r): r for r in bathy.eigenrays(ZS_EX, ZR_EX, *B.profile(pe[k], buoys[n]))}
    th_d = np.radians(rays["D"]["angle"])
    dop = {f: -FC * rr / C * (np.cos(np.radians(r["angle"])) - np.cos(th_d)) / np.cos(th_d)
           for f, r in rays.items() if f != "D"}
    shift_rows.append((n, T["k"][k], tr, lags[j], NUS[i], NUS[prof > -1][[0, -1]], rr, dop))
fig.tight_layout()
for n, kk, tr, lag, nu, flat, rr, dop in shift_rows:
    print(f"{n} ping {kk}: ridge echo delay {tr * 1e3:.3f} ms; correlation peak at {lag * 1e3:.3f} ms, {nu:+.0f} Hz; "
          f"within 1 dB of the peak for shifts {flat[0]:+.0f} to {flat[1]:+.0f} Hz; range rate {rr:+.2f} m/s "
          f"= {-FC * rr / C:+.1f} Hz common Doppler")
    print("   predicted shift over the direct path: " + ", ".join(f"{f} {v:+.3f} Hz" for f, v in dop.items()))


**What the frequency shift tells us: nothing per path, on this data.**

- **The correlation is a ridge, not a peak.** Delay and frequency shift trade along the white line at
  1 / sweep rate = 10 µs per Hz. Shifting a rising chirp up in frequency is the same as moving it earlier, so
  an LFM chirp cannot tell a frequency shift ν from a time shift ν/k. The correlation stays within 1 dB of its
  maximum over ~500 Hz of shifts. Its position along the ridge is set by noise, so the "best" shift means
  nothing. Only one combination is measured, the delay at zero shift, which is the echo delay of section 3.
- **The shifts to look for are far smaller.**
  - The common Doppler of all paths is the pinger's range rate: 23 Hz per m/s at 35 kHz (printed above).
    `detect.py` estimates it per ping from the pulse's time scale (`rr_dop`), but overlapping multipath biases
    that by 0.5–0.8 m/s (MAD against the arrival sequence). The track therefore uses the arrival sequence
    (`eta_track`) instead.
  - The extra shift of an echo over the direct path, from the geometry, is under 0.1 Hz (printed above).
    Surface wave motion could add a fraction of a Hz on paths that touch the surface.
  - A 100 ms pulse resolves 10 Hz, and the LFM coupling hides even that.
- **Where path information can still come from:** delays (sections 3–6) and how stable each path is from one
  ping to the next. Surface paths should decorrelate with wave motion; bottom paths should not.


## 4. When the reflections appear: echograms over the whole run

Each column is one ping (emission time on the buoy clock); each row is the delay behind the direct
path. Colour shows contrast, meaning the power over its median at that delay across the run. This removes the
reverberation tail that decays with delay, so a discrete arrival stands out wherever it is. The 20 % weakest pings are dropped. Cyan dots mark
the late arrival tracked by Viterbi in `multipath_range.track_ridge` (geometry-free, inside its segments).
The top strip shows the pinger course from the track, so the zig-zag legs are visible.

In [ ]:
def contrast(n, smooth=5):
    d = arr[n]
    _, As = mp.echogram(d, smooth)
    ok = d["ok"] & (T["snr"][:, col[n]] > np.nanpercentile(T["snr"][d["ok"], col[n]], 20))
    bg = np.median(As[ok], 0) + 1e-9
    return 10 * np.log10(As / bg + 1e-9), ok


fig, axs = plt.subplots(3, 1, figsize=(16, 11), sharex=True, gridspec_kw=dict(height_ratios=[0.6, 2, 2]))
axs[0].plot(t_ping, course, ".", ms=2, color="k")
axs[0].set_ylabel("course [deg]\n(CCW from east)")
axs[0].set_ylim(0, 360)
for name, t0, t1 in runs:
    axs[0].axvspan(t0, t1, color="C0" if name == "run1" else "C1", alpha=0.08)
    axs[0].text(max(t0, t_ping[0]) + 5, 300, name)
axs[0].set_xlim(t_ping[0] - 5, t_ping[-1] + 5)
for ax, n in zip(axs[1:], mp.BUOYS):
    G, ok = contrast(n)
    pc = ax.pcolormesh(t_ping[ok], arr[n]["t"] * 1e3, G[ok].T, vmin=-3, vmax=12, cmap="magma", shading="nearest")
    seg = M["mp_ridge_seg"][:, col[n]]
    ax.plot(t_ping[seg >= 0], M["mp_tau_ridge"][seg >= 0, col[n]] * 1e3, ".", color="c", ms=2)
    ax.set_ylim(0, 4)
    ax.set_ylabel("delay after direct [ms]")
    ax.set_title(f"{n}: contrast of the compressed pulse (5-ping incoherent mean)")
    fig.colorbar(pc, ax=ax, label="dB over median", pad=0.01)
axs[-1].set_xlabel("buoy time [s]")
fig.tight_layout()

### Removing the range: delay × range

In the far field every image delay is $K/R$, where $K = \left[(z_\mathrm{img}-z_r)^2-(z_s-z_r)^2\right]/(2c)$
depends only on the depths. Plotting $\tau R$ (R from the TOA track) should therefore turn each reflection
path into a horizontal line, and averaging over the run gives a "K-spectrum" whose peaks are the
path families. The dots on the right are the ray-traced families for the example depths
of section 2, and they move with the bottom profile under each ping.

In [ ]:
Kg = np.linspace(0.0, 1.6, 321)
kk_pred = {n: np.flatnonzero(arr[n]["ok"] & np.isfinite(pe[:, 0]))[::4] for n in mp.BUOYS}
pred_ex = {n: np.array([bathy.family_delays(ZS_EX, ZR_EX, *B.profile(pe[k], buoys[n]), FAMS, C)
                        for k in kk_pred[n]]) for n in mp.BUOYS}
fig, axs = plt.subplots(2, 2, figsize=(16, 9), gridspec_kw=dict(width_ratios=[4, 1.2]), sharey=True)
for (ax, ax2), n in zip(axs, mp.BUOYS):
    d = arr[n]
    A, _ = mp.echogram(d, smooth=1)
    G, ok = contrast(n, smooth=1)
    ok &= np.isfinite(R[n])
    GK = np.full((len(A), len(Kg)), np.nan)
    PK = np.full((len(A), len(Kg)), np.nan)
    for k in np.flatnonzero(ok):
        GK[k] = np.interp(Kg / R[n][k], d["t"], G[k], right=np.nan)
        PK[k] = np.interp(Kg / R[n][k], d["t"], A[k], right=np.nan)
    from scipy.ndimage import uniform_filter1d
    GKs = uniform_filter1d(np.nan_to_num(GK[ok]), 5, axis=0)
    ax.pcolormesh(t_ping[ok], Kg, GKs.T, vmin=-3, vmax=10, cmap="magma", shading="nearest")
    seg = M["mp_ridge_seg"][:, col[n]]
    ax.plot(t_ping[seg >= 0], M["mp_tau_ridge"][seg >= 0, col[n]] * R[n][seg >= 0], ".", color="c", ms=2)
    ax.set_ylabel(r"$\tau R$ [m s]")
    ax.set_title(f"{n}: echogram against tau x R")
    prof = np.nanmedian(PK[ok], 0)
    ax2.semilogx(prof, Kg, "k")
    for i, f in enumerate(FAMS):
        Kp = pred_ex[n][:, i] * R[n][kk_pred[n]]
        ax2.plot(np.full(len(Kp), 2e-3 * 1.6 ** i), Kp, ".", color=f"C{i}", ms=3, label=f)
    ax2.set_xlim(1e-3, 1.5)
    ax2.set_title("median over the run")
    ax2.legend(fontsize=7, ncol=2, loc="upper right")
axs[0, 0].set_ylim(0, 1.6)
axs[-1, 0].set_xlabel("buoy time [s]")
axs[-1, 1].set_xlabel("power rel. direct")
fig.tight_layout()

## 5. When is the late arrival there?

The tracked ridge's per-ping contrast and where on the track it is detected, plus how often it is
detected for each range and water depth under the pinger. Also the energy of the early
cluster (0.05–0.6 ms; mostly surface, plus first-order bottom paths) relative to the direct path.

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3)
ax0 = fig.add_subplot(gs[0, :])
for i, n in enumerate(mp.BUOYS):
    con = M["mp_ridge_contrast"][:, col[n]]
    seg = M["mp_ridge_seg"][:, col[n]]
    ax0.semilogy(t_ping, con, ".", ms=2, color=f"C{i}", alpha=0.3)
    ax0.semilogy(t_ping[seg >= 0], con[seg >= 0], ".", ms=4, color=f"C{i}", label=f"{n} (in a segment)")
ax0.axhline(3, color="k", lw=0.8, ls="--")
ax0.set_ylabel("ridge contrast")
ax0.set_xlabel("buoy time [s]")
ax0.legend(fontsize=8)
ax1 = fig.add_subplot(gs[1, :])
i0 = int(round(-mp.WIN[0] * arr["Rx902"]["fs"]))
for i, n in enumerate(mp.BUOYS):
    d = arr[n]
    A, _ = mp.echogram(d, smooth=1)
    early = (d["t"] > 0.05e-3) & (d["t"] < 0.6e-3)
    direct = np.abs(d["t"]) < 0.05e-3
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = 10 * np.log10(A[:, early].sum(1) / A[:, direct].sum(1))
    ratio[~d["ok"]] = np.nan
    ax1.plot(t_ping, ratio, ".", ms=2, color=f"C{i}", alpha=0.4)
    from scipy.ndimage import median_filter
    ax1.plot(t_ping, median_filter(np.nan_to_num(ratio, nan=np.nanmedian(ratio)), 21), color=f"C{i}", label=n)
ax1.set_ylabel("early cluster / direct [dB]")
ax1.set_xlabel("buoy time [s]")
ax1.legend(fontsize=8)
for i, n in enumerate(mp.BUOYS):
    seg = M["mp_ridge_seg"][:, col[n]]
    ok = arr[n]["ok"] & np.isfinite(R[n])
    ax = fig.add_subplot(gs[2, i])
    ax.plot(pe[ok, 0], pe[ok, 1], ".", color="0.8", ms=2)
    s = ax.scatter(pe[seg >= 0, 0], pe[seg >= 0, 1], c=(M["mp_tau_ridge"][:, col[n]] * R[n])[seg >= 0],
                   s=4, cmap="plasma", vmin=0.1, vmax=0.9)
    ax.plot(*buoys[n], "^", color="C3")
    ax.set_aspect("equal")
    ax.set_title(f"{n}: where the ridge is tracked")
    fig.colorbar(s, ax=ax, label=r"$\tau R$ [m s]", shrink=0.8)
ax = fig.add_subplot(gs[2, 2])
for i, n in enumerate(mp.BUOYS):
    seg = M["mp_ridge_seg"][:, col[n]]
    ok = arr[n]["ok"] & np.isfinite(R[n])
    bins = np.arange(5.5, 10.0, 0.5)
    b = np.digitize(D_pinger[ok], bins)
    frac = [np.mean(seg[ok][b == j] >= 0) if (b == j).sum() > 10 else np.nan for j in range(1, len(bins))]
    ax.plot(0.5 * (bins[1:] + bins[:-1]), frac, "o-", color=f"C{i}", label=n)
ax.set_xlabel("water depth under the pinger [m]")
ax.set_ylabel("fraction of pings with ridge")
ax.legend(fontsize=8)
fig.tight_layout()
for n in mp.BUOYS:
    seg = M["mp_ridge_seg"][:, col[n]]
    ok = arr[n]["ok"] & np.isfinite(R[n])
    for q in range(seg.max() + 1):
        m = seg == q
        print(f"{n} segment {q}: {t_ping[m].min():5.0f}-{t_ping[m].max():5.0f} s, {m.sum():3d} pings, "
              f"R {np.nanmedian(R[n][m]):4.0f} m, tau {np.nanmedian(M['mp_tau_ridge'][m, col[n]]) * 1e3:.2f} ms, "
              f"tau R {np.nanmedian(M['mp_tau_ridge'][m, col[n]] * R[n][m]):.3f} m s, "
              f"depth under pinger {np.nanmedian(D_pinger[m]):.1f} m")

## 6. Can the bathymetry identify the reflections?

Hypothesis test. For each ridge segment, each path family and each ($z_s$, $z_r$) on a grid, compare the
observed ridge delay with the ray-traced one: $r = \log(\tau_\mathrm{obs}/\tau_\mathrm{pred})$. A correct
hypothesis has median $r \approx 0$ and small scatter within the segment. The hydrophone depth is
one value per buoy, so each segment may pick its own family but all segments share $(z_s, z_r)$.
The pinger depth is the same for both buoys.

The grid is cached in `out/multipath/family_grid_<rx>.npz` and recomputed (~3 min on 16 cores) when the
track, buoy positions, sound speed or bathymetry change.

In [ ]:
ZS = np.arange(1.0, 7.01, 0.5)
ZR = np.arange(6.0, 12.51, 0.5)
grid = {}
for n in mp.BUOYS:
    f = ROOT / "out" / "multipath" / f"family_grid_{n}.npz"
    seg = M["mp_ridge_seg"][:, col[n]]
    kk = np.flatnonzero((seg >= 0) & np.isfinite(pe[:, 0]))[::2]
    # recomputed whenever the track, buoys, sound speed, grids or bathymetry change (stamp.py)
    key = stamp.key(files=[bathy.BATHY_DIR / "emodnet_mean_vodice.tif"], code=bathy.__file__,
                    kk=kk, pe=pe[kk], buoy=buoys[n], ZS=ZS, ZR=ZR, FAMS=FAMS, C=C,
                    max_bounces=bathy.MAX_BOUNCES)
    hit = stamp.load(f, key)
    if hit is not None:
        P = hit["P"]
    else:
        P = bathy.family_delay_grid([B.profile(pe[k], buoys[n]) for k in kk], ZS, ZR, FAMS, C)
        stamp.save(f, key, kk=kk, P=P, ZS=ZS, ZR=ZR, FAMS=np.array(FAMS))
    grid[n] = dict(kk=kk, P=P, tau=M["mp_tau_ridge"][kk, col[n]], seg=seg[kk])


def segment_costs(g):
    # per segment: |median r| + scatter of r, for every (zs, zr, family)
    lr = np.log(g["tau"][:, None, None, None] / g["P"])
    out = {}
    for s in np.unique(g["seg"]):
        q = g["seg"] == s
        med = np.nanmedian(lr[q], 0)
        mad = 1.4826 * np.nanmedian(np.abs(lr[q] - med), 0)
        out[s] = (q.sum(), np.nan_to_num(np.abs(med) + mad, nan=9.0), mad)
    return out


cost = {}
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, n in zip(axs, mp.BUOYS):
    sc = segment_costs(grid[n])
    tot = sum(w * c.min(2) for w, c, _ in sc.values()) / sum(w for w, _, _ in sc.values())
    cost[n] = tot
    pc = ax.pcolormesh(ZR, ZS, tot, cmap="viridis_r", vmax=0.3, shading="nearest")
    a, b = np.unravel_index(np.argmin(tot), tot.shape)
    ax.plot(ZR[b], ZS[a], "r*", ms=12)
    ax.set_xlabel("hydrophone depth z_r [m]")
    ax.set_ylabel("pinger depth z_s [m]")
    ax.set_title(f"{n}: ridge misfit (best family per segment)")
    fig.colorbar(pc, ax=ax)
    print(f"{n}: best z_s {ZS[a]} m, z_r {ZR[b]} m, misfit {tot[a, b]:.3f} "
          f"(median over the grid {np.median(tot):.3f}); families per segment: "
          + ", ".join(FAMS[c[a, b].argmin()] for _, c, _ in sc.values()))
J = sum(cost[n].min(1) for n in mp.BUOYS)
axs[2].plot(ZS, J, "o-k")
axs[2].set_xlabel("pinger depth z_s [m] (shared)")
axs[2].set_ylabel("misfit, both buoys (z_r free per buoy)")
axs[2].set_title("joint")
fig.tight_layout()

Does the sloping bottom explain the ridge better than a plain $\tau \propto 1/R$? Compare the
within-segment scatter of $\log \tau R$ (flat-bottom far field) with the best ray-traced hypothesis:

In [ ]:
for n in mp.BUOYS:
    g = grid[n]
    Rk = R[n][g["kk"]]
    for s, (w, c, mad) in segment_costs(g).items():
        q = g["seg"] == s
        lk = np.log(g["tau"][q] * Rk[q])
        flat = 1.4826 * np.median(np.abs(lk - np.median(lk)))
        print(f"{n} segment {s} ({w:2d} pings): scatter 1/R model {flat * 100:4.1f} %, "
              f"best bathymetry hypothesis {np.nanmin(mad) * 100:4.1f} %, median hypothesis {np.nanmedian(mad) * 100:4.1f} %")

## 7. Phase 2: does the boat's own noise show multipath?

For broadband noise, a reflection delayed by τ puts a peak at lag τ in the autocorrelation (equivalently,
striations in the spectrogram). Below: whitened autocorrelation (0.5–20 kHz, 1 s windows) on each buoy, with
its median over time removed so that only lags that move are left,
while the boat is audible, with the ray-traced delays of the first families for a noise source at the
placeholder depth `[noise].src_depth` and the configured hydrophone depth along the phase 2 track.

In [ ]:
from scipy import signal

import recordings

t2 = np.genfromtxt(ROOT / "out" / "phase2" / "track.csv", delimiter=",", names=True)
zs_noise = cfg.noise["src_depth"]
buoy_rx = [r for r in cfg.receivers if r.kind == "buoy"]     # same order as mp.BUOYS
fig, axs = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
for ax, n, rx in zip(axs, mp.BUOYS, buoy_rx):
    rec = recordings.load(rx)
    fs = rec.fs
    ts = np.arange(cfg.noise["t0"], cfg.noise["t1"], 1.0)
    maxlag = int(0.006 * fs)
    sos = signal.butter(4, [500, 20000], "bandpass", fs=fs, output="sos")
    AC = []
    for t in ts:
        x, _ = rec.segment(t, t + 1.0, ch=0)
        x = signal.sosfiltfilt(sos, np.asarray(x).ravel())
        P = np.abs(np.fft.rfft(x, 2 * len(x))) ** 2
        f = np.fft.rfftfreq(2 * len(x), 1 / fs)
        P = np.where((f > 500) & (f < 20000), P / (signal.medfilt(P, 301) + 1e-30), 0)
        ac = np.fft.irfft(P)[:maxlag]
        AC.append(ac / ac[0])
    AC = np.array(AC)
    AC -= np.median(AC, 0)                    # drop what does not move: self-noise lines, the band-limit mainlobe
    lag = np.arange(maxlag) / fs * 1e3
    ax.pcolormesh(ts, lag, np.abs(AC).T, vmin=0, vmax=0.03, cmap="magma", shading="nearest")
    sel = np.flatnonzero((t2["t_buoy_s"] >= cfg.noise["t0"]) & np.isfinite(t2["boat_e"]))[::5]
    pr = np.array([bathy.family_delays(zs_noise, Z_RX[n], *B.profile((t2["boat_e"][k], t2["boat_n"][k]), buoys[n]),
                                       FAMS, C) for k in sel])
    for i, f_ in enumerate(FAMS):
        ax.plot(t2["t_buoy_s"][sel], pr[:, i] * 1e3, ".", color=f"C{i}", ms=4, label=f_)
    ax.set_ylim(0.05, 6)
    ax.set_ylabel("lag [ms]")
    ax.set_title(f"{n}: whitened noise autocorrelation minus its time median; dots: predicted family delays (z_s = {zs_noise} m)")
    ax.legend(ncol=6, fontsize=8, loc="upper right")
axs[-1].set_xlabel("buoy time [s]")
fig.tight_layout()

The predicted delays (dots) all lie below ~1.5 ms, and nothing moves with them. The features that drift
at 3–5 ms (Rx902, ~840 s and 980–1015 s) are too late to be reflections at these ranges. They are more likely
the harmonic spacing of machinery noise (a 200–330 Hz fundamental; not verified).

## 8. What the multipath data gives, and what it does not

**It gives:**

- **An independent check on the track's range scaling.** τR stays constant to 3–6 % within each ridge
  segment. That holds only if the TOA track's ranges to each buoy are right up to a constant factor, and the
  test needs no clocks, sound speed or depths. It still cannot tell apart the GPS and the shifted Rx01
  position (`notes/D_multipath.md`).
- **A pattern in time.** The late arrival needs the pinger over deeper water (> ~7.5 m). The early cluster is as
  strong as the direct path throughout, which is why detection must use the first-arrival rule
  (`detect.py`) and not the biggest peak.
- **With the chart rather than the multipath:** the pinger must be shallower than ~5.8 m (the shoal), and
  the hydrophones are ~3 m above the seabed.

**It does not give** range, pinger depth, hydrophone depth, buoy position or sound speed. One discrete path
per buoy cannot fix three unknowns ($z_s$, $z_r$, and which family it is), and the early cluster is not
resolved into separate paths.

**Next steps, using data that already exists:**

- Re-run the coherent early-cluster fit (`multipath_range.ping_profile`) with D taken from the chart
  (5–14 m, per ping, from `bathy.Bathymetry.profile`) and not from the 12–45 m grid. The S, B and SB paths
  sit inside 0.6 ms, and the earlier failure may come partly from the grid.
- If the organisers have the actual hydrophone depth, or the pinger's tow depth setting, either one turns
  the late ridge into a family identification and then into a depth or range constraint.
- Higher-resolution bathymetry (a harbour chart or survey) than the ~100 m EMODnet cells: the multi-bounce
  delays depend on the shoal's shape at tens of metres.